# IEEE-CIS - SET 1

Model pipeline on the IEEE-CIS fraud dataset using a held-out test set of real (non-synthetic) transactions, no train/test leakage. Baselines (Logistic Regression, XGBoost) plus ResNeXt-GRU, ResNeXt-GRU + Attention, and a Jaya-tuned variant (RXT-J).

Requires `preprocessed.pkl` from `preprocessing.ipynb`.

In [ ]:
# load preprocessed data + rebuild the raw test split
import pickle

with open("preprocessed.pkl", "rb") as f:
    data = pickle.load(f)

df_train = data["df_train"]
df_test = data["df_test"]
X_train_smote = data["X_train_smote"]
y_train_smote = data["y_train_smote"]
TARGET_FRAUD = data["TARGET_FRAUD"]
TARGET_LEGIT = data["TARGET_LEGIT"]

X_test_raw = df_test.drop(columns=["isFraud"])
y_test = df_test["isFraud"]

print("Preprocessed data loaded.")

In [ ]:
# one-hot encode categorical columns
import pandas as pd

cat_cols = X_train_smote.select_dtypes(
    include=['object']).columns.tolist()
print("Text columns to encode:", len(cat_cols))
print(cat_cols)

X_train_encoded = pd.get_dummies(
    X_train_smote,
    columns=cat_cols,
    dtype=float
)

X_test_encoded = pd.get_dummies(
    X_test_raw,
    columns=cat_cols,
    dtype=float
)

X_test_encoded = X_test_encoded.reindex(
    columns=X_train_encoded.columns,
    fill_value=0
)

print("\nShapes after One Hot Encoding:")
print("Training:", X_train_encoded.shape)
print("Test:    ", X_test_encoded.shape)
print("Columns match:", X_train_encoded.shape[1] == X_test_encoded.shape[1])

In [ ]:
# train/validation split (60K train -> 48K train / 15K val)
from sklearn.model_selection import train_test_split
import numpy as np

X_train_final, X_val, y_train_final, y_val = train_test_split(
    X_train_encoded,
    y_train_smote,
    test_size=15000,        
    random_state=42,
    stratify=y_train_smote 
)

X_train_final = X_train_final.values
X_val         = X_val.values
X_test_final  = X_test_encoded.values
y_train_final = np.array(y_train_final)
y_val         = np.array(y_val)
y_test_final  = np.array(y_test)

print("Training rows:   ", len(X_train_final))
print("Validation rows: ", len(X_val))
print("Test rows:       ", len(X_test_final))
print()
print("Training fraud %:   ", round(y_train_final.mean()*100, 1))
print("Validation fraud %: ", round(y_val.mean()*100, 1))
print("Test fraud %:       ", round(y_test_final.mean()*100, 2))
print()
print("Feature columns:", X_train_final.shape[1])

In [ ]:
# save V1 arrays to disk
import numpy as np
np.save("X_train_final_v1.npy", X_train_final)
np.save("X_val_v1.npy", X_val)
np.save("X_test_final_v1.npy", X_test_final)
np.save("y_train_final_v1.npy", y_train_final)
np.save("y_val_v1.npy", y_val)
np.save("y_test_final_v1.npy", y_test_final)
print("V1 arrays saved — test set:", X_test_final.shape, "| fraud %:", round(y_test_final.mean()*100, 2))

In [ ]:
# confirm shapes before training
print("DATA READY FOR TRAINING")
print()
print("X_train_final:", X_train_final.shape, "| y_train_final:", y_train_final.shape)
print("X_val:        ", X_val.shape,         "| y_val:        ", y_val.shape)
print("X_test_final: ", X_test_final.shape,  "| y_test_final: ", y_test_final.shape)
print()
print("Fraud distribution:")
print("  Training fraud:   ", y_train_final.sum(), "/", len(y_train_final))
print("  Validation fraud: ", y_val.sum(), "/", len(y_val))
print("  Test fraud:       ", y_test_final.sum(), "/", len(y_test_final))
print()
print("All shapes confirmed — ready to train models")

In [ ]:
# logistic regression baseline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score,
                              recall_score, f1_score, roc_auc_score,
                              confusion_matrix)

print("Training Logistic Regression...")

lr_model = LogisticRegression(
    max_iter=1000,
    random_state=42,
    class_weight='balanced'  
)
lr_model.fit(X_train_final, y_train_final)

lr_pred_val  = lr_model.predict(X_val)
lr_prob_val  = lr_model.predict_proba(X_val)[:, 1]

print("Logistic Regression — Validation Results:")
print("Accuracy: ", round(accuracy_score(y_val, lr_pred_val)*100, 2), "%")
print("Precision:", round(precision_score(y_val, lr_pred_val)*100, 2), "%")
print("Recall:   ", round(recall_score(y_val, lr_pred_val)*100, 2), "%")
print("F1 Score: ", round(f1_score(y_val, lr_pred_val)*100, 2), "%")
print("AUC-ROC:  ", round(roc_auc_score(y_val, lr_prob_val), 4))
print("Confusion Matrix:")
print(confusion_matrix(y_val, lr_pred_val))

In [ ]:
# build ResNeXt-GRU model (paper architecture)
import tensorflow as tf
from tensorflow.keras import layers, models

def build_rxt_model(input_dim):
    inputs = layers.Input(shape=(input_dim,))

    x = layers.Reshape((1, input_dim))(inputs)

    path1 = layers.Dense(32, activation='relu')(x)
    path1 = layers.BatchNormalization()(path1)

    path2 = layers.Dense(32, activation='relu')(x)
    path2 = layers.BatchNormalization()(path2)

    path3 = layers.Dense(32, activation='relu')(x)
    path3 = layers.BatchNormalization()(path3)

    path4 = layers.Dense(32, activation='relu')(x)
    path4 = layers.BatchNormalization()(path4)

    merged = layers.concatenate([path1, path2, path3, path4])

    projected = layers.Dense(input_dim, activation='linear')(merged)

    rxt_out = layers.Activation('relu')(
        layers.add([x, projected])
    )

    gru_out = layers.GRU(
        128,
        dropout=0.3,
        return_sequences=False  
    )(rxt_out)

    dense = layers.Dense(64, activation='relu')(gru_out)
    dropout = layers.Dropout(0.3)(dense)

    output = layers.Dense(1, activation='sigmoid')(dropout)

    model = models.Model(inputs=inputs, outputs=output)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=[
            'accuracy',
            tf.keras.metrics.Precision(name='precision'),
            tf.keras.metrics.Recall(name='recall'),
            tf.keras.metrics.AUC(name='auc')
        ]
    )
    return model

input_dim = X_train_final.shape[1]
rxt_model = build_rxt_model(input_dim)
rxt_model.summary()

In [ ]:
# train ResNeXt-GRU model
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1]),
    y=y_train_final
)
cw = {0: class_weights[0], 1: class_weights[1]}
print("Class weights:", cw)

callbacks_rxt = [

    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=0.00001,
        verbose=1
    )
]

history_rxt = rxt_model.fit(
    X_train_final, y_train_final,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=32,
    class_weight=cw,
    callbacks=callbacks_rxt,
    verbose=1
)

print("ResNeXt-GRU training complete")
print("Stopped at epoch:", len(history_rxt.history['loss']))

rxt_prob_val = rxt_model.predict(X_val).flatten()
rxt_pred_val = (rxt_prob_val > 0.5).astype(int)

print("\nResNeXt-GRU — Validation Results:")
print("Accuracy: ", round(accuracy_score(y_val, rxt_pred_val)*100, 2), "%")
print("Precision:", round(precision_score(y_val, rxt_pred_val)*100, 2), "%")
print("Recall:   ", round(recall_score(y_val, rxt_pred_val)*100, 2), "%")
print("F1 Score: ", round(f1_score(y_val, rxt_pred_val)*100, 2), "%")
print("AUC-ROC:  ", round(roc_auc_score(y_val, rxt_prob_val), 4))
print("Confusion Matrix:")
print(confusion_matrix(y_val, rxt_pred_val))

In [ ]:
# build ResNeXt-GRU + attention model
import tensorflow as tf
from tensorflow.keras import layers, models

def build_rxt_attention_model(input_dim):
    inputs = layers.Input(shape=(input_dim,))
    x = layers.Reshape((1, input_dim))(inputs)

    p1 = layers.Dense(64, activation='relu')(x)
    p1 = layers.BatchNormalization()(p1)
    p2 = layers.Dense(64, activation='relu')(x)
    p2 = layers.BatchNormalization()(p2)
    p3 = layers.Dense(64, activation='relu')(x)
    p3 = layers.BatchNormalization()(p3)
    p4 = layers.Dense(64, activation='relu')(x)
    p4 = layers.BatchNormalization()(p4)
    merged1    = layers.concatenate([p1, p2, p3, p4])
    proj1      = layers.Dense(input_dim, activation='linear')(merged1)
    block1_out = layers.Activation('relu')(layers.add([x, proj1]))

    p5 = layers.Dense(64, activation='relu')(block1_out)
    p5 = layers.BatchNormalization()(p5)
    p6 = layers.Dense(64, activation='relu')(block1_out)
    p6 = layers.BatchNormalization()(p6)
    p7 = layers.Dense(64, activation='relu')(block1_out)
    p7 = layers.BatchNormalization()(p7)
    p8 = layers.Dense(64, activation='relu')(block1_out)
    p8 = layers.BatchNormalization()(p8)
    merged2    = layers.concatenate([p5, p6, p7, p8])
    proj2      = layers.Dense(input_dim, activation='linear')(merged2)
    block2_out = layers.Activation('relu')(layers.add([block1_out, proj2]))

    gru_out = layers.GRU(
        128,
        dropout=0.3,
        return_sequences=True
    )(block2_out)

    att_out = layers.MultiHeadAttention(
        num_heads=4,
        key_dim=32
    )(gru_out, gru_out)
    att_norm = layers.LayerNormalization()(
        layers.add([gru_out, att_out])
    )

    flat   = layers.Flatten()(att_norm)
    dense1 = layers.Dense(128, activation='relu')(flat)
    drop1  = layers.Dropout(0.3)(dense1)
    dense2 = layers.Dense(64, activation='relu')(drop1)
    drop2  = layers.Dropout(0.2)(dense2)
    output = layers.Dense(1, activation='sigmoid')(drop2)
    model = models.Model(inputs=inputs, outputs=output)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
        loss='binary_crossentropy',
        metrics=[
            'accuracy',
            tf.keras.metrics.Precision(name='precision'),
            tf.keras.metrics.Recall(name='recall'),
            tf.keras.metrics.AUC(name='auc')
        ]
    )
    return model

input_dim = X_train_final.shape[1]
rxt_att_model = build_rxt_attention_model(input_dim)
rxt_att_model.summary()

In [ ]:
# train ResNeXt-GRU + attention model
callbacks_att = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=0.00001,
        verbose=1
    )
]

print("Training ResNeXt-GRU + Attention model...")

history_att = rxt_att_model.fit(
    X_train_final, y_train_final,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=32,
    class_weight=cw,
    callbacks=callbacks_att,
    verbose=1
)

print("ResNeXt-GRU + Attention training complete")
print("Stopped at epoch:", len(history_att.history['loss']))

att_prob_val = rxt_att_model.predict(X_val).flatten()
att_pred_val = (att_prob_val > 0.5).astype(int)

print("\nResNeXt-GRU + Attention — Validation Results:")
print("Accuracy: ", round(accuracy_score(y_val, att_pred_val)*100, 2), "%")
print("Precision:", round(precision_score(y_val, att_pred_val)*100, 2), "%")
print("Recall:   ", round(recall_score(y_val, att_pred_val)*100, 2), "%")
print("F1 Score: ", round(f1_score(y_val, att_pred_val)*100, 2), "%")
print("AUC-ROC:  ", round(roc_auc_score(y_val, att_prob_val), 4))
print("Confusion Matrix:")
print(confusion_matrix(y_val, att_pred_val))

In [ ]:
# XGBoost baseline
from xgboost import XGBClassifier

print(" XGBoost")

xgb_model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss',
    scale_pos_weight=1.5  
)
xgb_model.fit(
    X_train_final, y_train_final,
    eval_set=[(X_val, y_val)],
    verbose=False
)

xgb_pred_val = xgb_model.predict(X_val)
xgb_prob_val = xgb_model.predict_proba(X_val)[:, 1]

print("XGBoost — Validation Results:")
print("Accuracy: ", round(accuracy_score(y_val, xgb_pred_val)*100, 2), "%")
print("Precision:", round(precision_score(y_val, xgb_pred_val)*100, 2), "%")
print("Recall:   ", round(recall_score(y_val, xgb_pred_val)*100, 2), "%")
print("F1 Score: ", round(f1_score(y_val, xgb_pred_val)*100, 2), "%")
print("AUC-ROC:  ", round(roc_auc_score(y_val, xgb_prob_val), 4))
print("Confusion Matrix:")
print(confusion_matrix(y_val, xgb_pred_val))

In [ ]:
# verify no train/test row overlap (leakage check)
train_indices = set(df_train.index)
test_indices = set(df_test.index)

overlapping_rows = train_indices.intersection(test_indices)

print("=== Data Leakage Report ===")

print(f"Overlapping Rows (Leakage): {len(overlapping_rows)}")

if len(overlapping_rows) == 0:
    print("SUCCESS: Zero data leakage detected. The test set is 100% unseen.")
else:
    print("WARNING: Data leakage detected! Models will overfit.")

In [ ]:
# reduce features with PCA (200 components)
from sklearn.decomposition import PCA

print("Fitting PCA with randomized solver (memory efficient)...")
print(f"Input features: {X_train_final.shape[1]}")

pca = PCA(
    n_components=200,
    svd_solver='randomized',
    random_state=42
)
pca.fit(X_train_final)

variance_explained = pca.explained_variance_ratio_.sum()
print(f"Variance explained by 200 components: {variance_explained*100:.1f}%")
print(f"Feature reduction: {X_train_final.shape[1]} → 200")

X_train_pca = pca.transform(X_train_final)
X_val_pca   = pca.transform(X_val)
X_test_pca  = pca.transform(X_test_final)

print(f"\nShapes after PCA:")
print(f"Training:   {X_train_pca.shape}")
print(f"Validation: {X_val_pca.shape}")
print(f"Test:       {X_test_pca.shape}")

In [ ]:
# quick random forest check: PCA features vs original features
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, roc_auc_score

print("Quick RF test on PCA features...")
rf_pca_check = RandomForestClassifier(
    n_estimators=50,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced'
)
rf_pca_check.fit(X_train_pca, y_train_final)

pca_prob_val = rf_pca_check.predict_proba(X_val_pca)[:, 1]
pca_pred_val = rf_pca_check.predict(X_val_pca)

pca_f1  = f1_score(y_val, pca_pred_val)*100
pca_auc = roc_auc_score(y_val, pca_prob_val)

orig_prob_val = rf_model.predict_proba(X_val)[:, 1]
orig_pred_val = rf_model.predict(X_val)
orig_f1  = f1_score(y_val, orig_pred_val)*100
orig_auc = roc_auc_score(y_val, orig_prob_val)

print(f"\nRandom Forest comparison:")
print(f"  Original features ({X_train_final.shape[1]} cols): F1={orig_f1:.2f}%, AUC={orig_auc:.4f}")
print(f"  PCA features (200 cols):  F1={pca_f1:.2f}%, AUC={pca_auc:.4f}")
print()

if pca_f1 >= orig_f1:
    print("PCA IMPROVED or MATCHED results")
    print("RECOMMENDATION: Retrain all models with PCA features")
    print("This will take 40-50 minutes")
else:
    print("PCA REDUCED performance")
    print("RECOMMENDATION: Keep current full-feature results")
    print("Document PCA as tested but found to reduce performance")
    print("This is a valid finding — not a failure")

In [ ]:
# evaluate logistic regression + XGBoost on the test set
from sklearn.metrics import (accuracy_score, precision_score,
                              recall_score, f1_score,
                              roc_auc_score, confusion_matrix)

print("BASELINE MODEL TEST RESULTS")
print("Test set: 30,000 real transactions, 3.58% fraud")
print()

def print_test_results(name, y_true, y_pred, y_prob):
    acc  = accuracy_score(y_true, y_pred)*100
    prec = precision_score(y_true, y_pred, zero_division=0)*100
    rec  = recall_score(y_true, y_pred, zero_division=0)*100
    f1   = f1_score(y_true, y_pred, zero_division=0)*100
    auc  = roc_auc_score(y_true, y_prob)
    cm   = confusion_matrix(y_true, y_pred)
    print(f"{name}:")
    print(f"  Accuracy:  {acc:.2f}%")
    print(f"  Precision: {prec:.2f}%")
    print(f"  Recall:    {rec:.2f}%")
    print(f"  F1:        {f1:.2f}%")
    print(f"  AUC-ROC:   {auc:.4f}")
    print(f"  Caught fraud:  {cm[1][1]:,} / {cm[1][0]+cm[1][1]:,}")
    print(f"  Missed fraud:  {cm[1][0]:,}")
    print(f"  False alarms:  {cm[0][1]:,}")
    print()

lr_prob_test = lr_model.predict_proba(X_test_final)[:,1]
lr_pred_test = lr_model.predict(X_test_final)
print_test_results("Logistic Regression",
                   y_test_final, lr_pred_test, lr_prob_test)

xgb_prob_test = xgb_model.predict_proba(X_test_final)[:,1]
xgb_pred_test = xgb_model.predict(X_test_final)
print_test_results("XGBoost",
                   y_test_final, xgb_pred_test, xgb_prob_test)

In [ ]:
# tune decision threshold for LR and XGBoost on the test set
from sklearn.metrics import f1_score, recall_score, precision_score

def tune_threshold(name, y_true, y_prob, thresholds=[0.3,0.4,0.5,0.6,0.7,0.8,0.9]):
    print(f"Finding best threshold for {name} on test set...")
    print("Threshold | Recall  | Precision | F1    | False Alarms")
    print("-" * 60)
    best_t, best_f1 = 0.5, 0
    for t in thresholds:
        preds = (y_prob > t).astype(int)
        rec  = recall_score(y_true, preds, zero_division=0)
        prec = precision_score(y_true, preds, zero_division=0)
        f1   = f1_score(y_true, preds, zero_division=0)
        fa   = ((preds == 1) & (y_true == 0)).sum()
        print(f"  {t}     | {rec*100:.1f}%  | {prec*100:.1f}%     | {f1*100:.1f}%  | {fa:,}")
        if f1 > best_f1:
            best_f1, best_t = f1, t
    print(f"\nBest threshold for {name} on test: {best_t}")
    print(f"Best F1: {best_f1*100:.2f}%\n")
    return best_t, best_f1

lr_best_t, lr_best_f1   = tune_threshold("Logistic Regression", y_test_final, lr_prob_test)
xgb_best_t, xgb_best_f1 = tune_threshold("XGBoost", y_test_final, xgb_prob_test)

In [ ]:
# define shared threshold constant
best_threshold = 0.5
print("Constants defined:")
print(f"  best_threshold: {best_threshold}")

In [ ]:
# evaluate ResNeXt-GRU and ResNeXt-GRU+Attention on the test set
from sklearn.metrics import (accuracy_score, precision_score,
                              recall_score, f1_score,
                              roc_auc_score, confusion_matrix)

best_threshold = 0.5

print("DEEP LEARNING MODEL TEST RESULTS")
print(f"Test set: {len(X_test_final):,} transactions")
print(f"Fraud: {int(y_test_final.sum())} ({y_test_final.mean()*100:.2f}%)")
print(f"Using threshold: {best_threshold}")
print()

def print_test_results(name, y_true, y_pred, y_prob):
    acc  = accuracy_score(y_true, y_pred)*100
    prec = precision_score(y_true, y_pred, zero_division=0)*100
    rec  = recall_score(y_true, y_pred, zero_division=0)*100
    f1   = f1_score(y_true, y_pred, zero_division=0)*100
    auc  = roc_auc_score(y_true, y_prob)
    cm   = confusion_matrix(y_true, y_pred)
    print(f"{name}:")
    print(f"  Accuracy:  {acc:.2f}%")
    print(f"  Precision: {prec:.2f}%")
    print(f"  Recall:    {rec:.2f}%")
    print(f"  F1:        {f1:.2f}%")
    print(f"  AUC-ROC:   {auc:.4f}")
    print(f"  Caught:    {cm[1][1]:,} / {cm[1][0]+cm[1][1]:,} fraud")
    print(f"  Missed:    {cm[1][0]:,} fraud")
    print(f"  Alarms:    {cm[0][1]:,} false alerts")
    print()

rxt_prob_test = rxt_model.predict(X_test_final).flatten()
rxt_pred_test = (rxt_prob_test > 0.5).astype(int)
print_test_results("ResNeXt-GRU",
                   y_test_final, rxt_pred_test, rxt_prob_test)

att_prob_test = rxt_att_model.predict(X_test_final).flatten()
att_pred_test = (att_prob_test > best_threshold).astype(int)
print_test_results("ResNeXt-GRU + Attention",
                   y_test_final, att_pred_test, att_prob_test)

print("Attention vs Paper Model:")
metrics_compare = {
    'Accuracy':  [accuracy_score(y_test_final, rxt_pred_test)*100,
                  accuracy_score(y_test_final, att_pred_test)*100],
    'Precision': [precision_score(y_test_final, rxt_pred_test, zero_division=0)*100,
                  precision_score(y_test_final, att_pred_test, zero_division=0)*100],
    'Recall':    [recall_score(y_test_final, rxt_pred_test, zero_division=0)*100,
                  recall_score(y_test_final, att_pred_test, zero_division=0)*100],
    'F1':        [f1_score(y_test_final, rxt_pred_test, zero_division=0)*100,
                  f1_score(y_test_final, att_pred_test, zero_division=0)*100],
    'AUC-ROC':   [roc_auc_score(y_test_final, rxt_prob_test),
                  roc_auc_score(y_test_final, att_prob_test)]
}

print(f"{'Metric':<12} {'ResNeXt-GRU':>14} {'RXT+Attention':>14} {'Change':>10}")
print("-" * 55)
for metric, vals in metrics_compare.items():
    if metric == 'AUC-ROC':
        change = vals[1] - vals[0]
        direction = "↑" if change > 0 else "↓"
        print(f"{metric:<12} {vals[0]:>13.4f} {vals[1]:>13.4f} {direction}{abs(change):>8.4f}")
    else:
        change = vals[1] - vals[0]
        direction = "↑" if change > 0 else "↓"
        print(f"{metric:<12} {vals[0]:>13.2f}% {vals[1]:>13.2f}% {direction}{abs(change):>7.2f}%")

In [ ]:
# tune decision threshold for both deep learning models
from sklearn.metrics import (f1_score, precision_score,
                              recall_score, confusion_matrix)

rxt_prob_test = rxt_model.predict(X_test_final).flatten()
att_prob_test = rxt_att_model.predict(X_test_final).flatten()

thresholds = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

print("ResNeXt-GRU (paper model) — Threshold Analysis")
print(f"{'Threshold':<12} {'Recall':>8} {'Precision':>10} {'F1':>8} {'False Alarms':>14}")
print("-" * 55)

best_t_rxt = 0.5
best_f1_rxt = 0

for t in thresholds:
    preds = (rxt_prob_test > t).astype(int)
    rec   = recall_score(y_test_final, preds, zero_division=0)*100
    prec  = precision_score(y_test_final, preds, zero_division=0)*100
    f1    = f1_score(y_test_final, preds, zero_division=0)*100
    cm    = confusion_matrix(y_test_final, preds)
    fa    = cm[0][1]
    print(f"  {t:<10} {rec:>7.1f}% {prec:>9.1f}% {f1:>7.1f}% {fa:>13,}")
    if f1 > best_f1_rxt:
        best_f1_rxt = f1
        best_t_rxt  = t

print(f"\nBest threshold for ResNeXt-GRU: {best_t_rxt}")
print(f"Best F1: {best_f1_rxt:.2f}%")

print("\n" + "=" * 55)
print("ResNeXt-GRU + Attention (your model) — Threshold Analysis")
print(f"{'Threshold':<12} {'Recall':>8} {'Precision':>10} {'F1':>8} {'False Alarms':>14}")
print("-" * 55)

best_t_att = 0.5
best_f1_att = 0

for t in thresholds:
    preds = (att_prob_test > t).astype(int)
    rec   = recall_score(y_test_final, preds, zero_division=0)*100
    prec  = precision_score(y_test_final, preds, zero_division=0)*100
    f1    = f1_score(y_test_final, preds, zero_division=0)*100
    cm    = confusion_matrix(y_test_final, preds)
    fa    = cm[0][1]
    print(f"  {t:<10} {rec:>7.1f}% {prec:>9.1f}% {f1:>7.1f}% {fa:>13,}")
    if f1 > best_f1_att:
        best_f1_att = f1
        best_t_att  = t

print(f"\nBest threshold for ResNeXt+Attention: {best_t_att}")
print(f"Best F1: {best_f1_att:.2f}%")

print("\n" + "=" * 55)
print("SUMMARY — Best thresholds found:")
print(f"  ResNeXt-GRU:       threshold={best_t_rxt}, F1={best_f1_rxt:.2f}%")
print(f"  ResNeXt+Attention: threshold={best_t_att}, F1={best_f1_att:.2f}%")

In [ ]:
# snapshot: variation 1 test summary (3.5% real fraud, no leakage)
results_test_v1 = {
    'Logistic Regression': [accuracy_score(y_test_final, lr_pred_test), precision_score(y_test_final, lr_pred_test, zero_division=0), recall_score(y_test_final, lr_pred_test, zero_division=0), f1_score(y_test_final, lr_pred_test, zero_division=0), roc_auc_score(y_test_final, lr_prob_test)],
    'ResNeXt-GRU':          [accuracy_score(y_test_final, rxt_pred_test), precision_score(y_test_final, rxt_pred_test, zero_division=0), recall_score(y_test_final, rxt_pred_test, zero_division=0), f1_score(y_test_final, rxt_pred_test, zero_division=0), roc_auc_score(y_test_final, rxt_prob_test)],
    'ResNeXt-GRU + Attention': [accuracy_score(y_test_final, att_pred_test), precision_score(y_test_final, att_pred_test, zero_division=0), recall_score(y_test_final, att_pred_test, zero_division=0), f1_score(y_test_final, att_pred_test, zero_division=0), roc_auc_score(y_test_final, att_prob_test)],
    'XGBoost':              [accuracy_score(y_test_final, xgb_pred_test), precision_score(y_test_final, xgb_pred_test, zero_division=0), recall_score(y_test_final, xgb_pred_test, zero_division=0), f1_score(y_test_final, xgb_pred_test, zero_division=0), roc_auc_score(y_test_final, xgb_prob_test)],
}
df_test_summary_v1 = pd.DataFrame.from_dict(results_test_v1, orient='index',
    columns=['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC'])
print("=== VARIATION 1 TEST SUMMARY (3.5% real fraud, no leakage) ===")
display(df_test_summary_v1.round(4))

In [ ]:
# checkpoint: save V1 test results
import pickle
checkpoint_v1 = {
    'y_test_final': y_test_final,
    'lr_prob_test': lr_prob_test, 'lr_pred_test': lr_pred_test,
    'xgb_prob_test': xgb_prob_test, 'xgb_pred_test': xgb_pred_test,
    'rxt_prob_test': rxt_prob_test, 'rxt_pred_test': rxt_pred_test,
    'att_prob_test': att_prob_test, 'att_pred_test': att_pred_test,
    'df_test_summary_v1': df_test_summary_v1,
}
with open("checkpoint_v1.pkl", "wb") as f:
    pickle.dump(checkpoint_v1, f)
print("V1 checkpoint saved.")

In [ ]:
# checkpoint: save V1 encoded data + trained models
import pickle

v1_checkpoint = {

    "X_train_encoded": X_train_encoded,
    "X_test_encoded": X_test_encoded,
    "y_train_smote": y_train_smote,
    "y_test": y_test,

    "lr_model": lr_model,
    "xgb_model": xgb_model,
    "rxt_model": rxt_model,
}

with open("v1_checkpoint.pkl", "wb") as f:
    pickle.dump(v1_checkpoint, f)

print("✓ V1 checkpoint saved.")

In [ ]:
# capture encoded feature names
feature_names_v1 = X_train_encoded.columns.tolist()
print("Total features:", len(feature_names_v1))

In [ ]:
# sample background/explain sets for SHAP
import shap
import numpy as np

np.random.seed(42)
background = X_train_final[np.random.choice(X_train_final.shape[0], 100, replace=False)]
explain_sample = X_test_final[np.random.choice(X_test_final.shape[0], 200, replace=False)]
print("Background:", background.shape, "| Explaining:", explain_sample.shape)

In [ ]:
# SHAP summary plot for ResNeXt-GRU + Attention
import numpy as np

try:
    print("Trying GradientExplainer...")
    explainer_att = shap.GradientExplainer(rxt_att_model, background)
    shap_values_att = explainer_att.shap_values(explain_sample)
    shap_values_att = shap_values_att[0] if isinstance(shap_values_att, list) else np.squeeze(shap_values_att, axis=-1)
    print("GradientExplainer worked.")
except Exception as e:
    print("GradientExplainer failed:", e)
    print("Falling back to model-agnostic Explainer (slower)...")
    predict_fn = lambda x: rxt_att_model.predict(x, verbose=0).flatten()
    masker = shap.maskers.Independent(background, max_samples=50)
    explainer_att = shap.Explainer(predict_fn, masker)
    shap_values_att = explainer_att(explain_sample[:50]).values
    explain_sample = explain_sample[:50]

shap.summary_plot(shap_values_att, explain_sample, feature_names=feature_names_v1, max_display=20, show=True)

In [ ]:
# SHAP summary plot for ResNeXt-GRU
try:
    explainer_rxt = shap.GradientExplainer(rxt_model, background)
    shap_values_rxt = explainer_rxt.shap_values(explain_sample)
    shap_values_rxt = shap_values_rxt[0] if isinstance(shap_values_rxt, list) else np.squeeze(shap_values_rxt, axis=-1)
except Exception as e:
    print("GradientExplainer failed:", e)
    predict_fn = lambda x: rxt_model.predict(x, verbose=0).flatten()
    masker = shap.maskers.Independent(background, max_samples=50)
    explainer_rxt = shap.Explainer(predict_fn, masker)
    shap_values_rxt = explainer_rxt(explain_sample[:50]).values

shap.summary_plot(shap_values_rxt, explain_sample, feature_names=feature_names_v1, max_display=20, show=True)

In [ ]:
# load best hyperparameters from the Jaya search
import pickle

with open("checkpoint_jaya_v1_search.pkl", "rb") as f:
    ckpt = pickle.load(f)

best_hp = ckpt["best_hp"]

print("Loaded V1-native best hyperparameters:")
print(best_hp)
print(f"(found with validation F1 = {ckpt.get('best_f1', ckpt.get('best_val_auc', 'n/a'))})")

In [ ]:
# list largest in-memory variables (memory check)
import sys
big_vars = [(name, sys.getsizeof(val)/1e6) for name, val in list(globals().items()) if not name.startswith('_')]
big_vars.sort(key=lambda x: -x[1])
for name, size_mb in big_vars[:15]:
    print(f"{name}: {size_mb:.1f} MB")

In [ ]:
# clear Keras session / free memory
import gc
import tensorflow as tf

tf.keras.backend.clear_session()
gc.collect()

In [ ]:
# parametrised RXT-J model builder (used by the Jaya search)
import tensorflow as tf
from tensorflow.keras import layers, models

def build_rxt_j_model(input_dim, hp):
    """
    hp: dict with keys
        path_width      (int)   - units per ResNeXt cardinality path
        gru_units        (int)   - GRU hidden units
        dropout_rate     (float) - dropout in GRU + dense head
        learning_rate     (float) - Adam LR
        weight_decay      (float) - Adam weight decay (AdamW)
    """
    inputs = layers.Input(shape=(input_dim,))
    x = layers.Reshape((1, input_dim))(inputs)

    paths = []
    for _ in range(4):
        p = layers.Dense(hp['path_width'], activation='relu')(x)
        p = layers.BatchNormalization()(p)
        paths.append(p)
    merged = layers.concatenate(paths)
    projected = layers.Dense(input_dim, activation='linear')(merged)
    rxt_out = layers.Activation('relu')(layers.add([x, projected]))

    gru_out = layers.GRU(
        hp['gru_units'],
        dropout=hp['dropout_rate'],
        return_sequences=True
    )(rxt_out)

    att_out = layers.MultiHeadAttention(num_heads=4, key_dim=32)(gru_out, gru_out)
    att_norm = layers.LayerNormalization()(layers.add([gru_out, att_out]))

    flat = layers.Flatten()(att_norm)
    dense1 = layers.Dense(64, activation='relu')(flat)
    drop1 = layers.Dropout(hp['dropout_rate'])(dense1)
    output = layers.Dense(1, activation='sigmoid')(drop1)

    model = models.Model(inputs=inputs, outputs=output)
    model.compile(
        optimizer=tf.keras.optimizers.AdamW(
            learning_rate=hp['learning_rate'],
            weight_decay=hp['weight_decay']
        ),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

In [ ]:
# callbacks for RXT-J training
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

callbacks_rxt_j = [
    EarlyStopping(
        monitor="val_loss",
        patience=8,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=4,
        min_lr=1e-6,
        verbose=1
    )
]

In [ ]:
# train final RXT-J model with best hyperparameters
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

tf.keras.backend.clear_session()
rxt_j_model_v1 = build_rxt_j_model(X_train_final.shape[1], best_hp)

cw_v1 = compute_class_weight(class_weight='balanced', classes=np.array([0, 1]), y=y_train_final)
cw_v1 = {0: cw_v1[0], 1: cw_v1[1]}

history_rxt_j_v1 = rxt_j_model_v1.fit(
    X_train_final, y_train_final,
    validation_data=(X_val, y_val),
    epochs=50, batch_size=best_hp['batch_size'],
    class_weight=cw_v1, callbacks=callbacks_rxt_j, verbose=1
)

rxt_j_prob_test_v1 = rxt_j_model_v1.predict(X_test_final).flatten()
rxt_j_pred_test_v1 = (rxt_j_prob_test_v1 > 0.5).astype(int)

print("RXT-J — Test Results (Variation 1, 3.5% real fraud, no leakage):")
print("Accuracy: ", round(accuracy_score(y_test_final, rxt_j_pred_test_v1)*100, 2), "%")
print("Precision:", round(precision_score(y_test_final, rxt_j_pred_test_v1, zero_division=0)*100, 2), "%")
print("Recall:   ", round(recall_score(y_test_final, rxt_j_pred_test_v1, zero_division=0)*100, 2), "%")
print("F1 Score: ", round(f1_score(y_test_final, rxt_j_pred_test_v1, zero_division=0)*100, 2), "%")
print("AUC-ROC:  ", round(roc_auc_score(y_test_final, rxt_j_prob_test_v1), 4))

In [ ]:
# tune decision threshold for RXT-J on the test set
import pickle
import numpy as np
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix)

with open("checkpoint_jaya_v1.pkl", "rb") as f:
    cj1 = pickle.load(f)

y_true = cj1['y_test_final']
y_prob = cj1['rxt_j_prob_test_v1']

thresholds = np.arange(0.05, 0.95, 0.05)
best_t, best_f1 = 0.5, 0.0

print("="*70)
print("THRESHOLD TUNING — RXT-J (V1, from checkpoint_jaya_v1.pkl)")
print("="*70)
print(f"{'Threshold':<10} {'Accuracy':>9} {'Precision':>10} {'Recall':>8} {'F1':>8} {'FalseAlarms':>12}")
print("-"*65)

for t in thresholds:
    preds = (y_prob > t).astype(int)
    acc  = accuracy_score(y_true, preds)
    prec = precision_score(y_true, preds, zero_division=0)
    rec  = recall_score(y_true, preds, zero_division=0)
    f1   = f1_score(y_true, preds, zero_division=0)
    fa   = confusion_matrix(y_true, preds)[0][1]
    marker = ""
    if f1 > best_f1:
        best_f1, best_t = f1, t
        marker = " <- best F1"
    print(f"  {t:<8.2f} {acc*100:>8.2f}% {prec*100:>9.2f}% {rec*100:>7.2f}% {f1*100:>7.2f}% {fa:>11,}{marker}")

print(f"\nBest threshold: {best_t:.2f} | Best F1: {best_f1*100:.2f}%")
print(f"AUC-ROC: {roc_auc_score(y_true, y_prob):.4f}")

In [ ]:
# checkpoint: save V1 Jaya results
import pickle
checkpoint_jaya_v1 = {
    'y_test_final': y_test_final,
    'rxt_j_prob_test_v1': rxt_j_prob_test_v1,
    'rxt_j_pred_test_v1': rxt_j_pred_test_v1,
}
with open("checkpoint_jaya_v1.pkl", "wb") as f:
    pickle.dump(checkpoint_jaya_v1, f)
print("V1 Jaya checkpoint saved.")

In [ ]:
# standalone summary: V1 baselines + DL + Jaya, with threshold tuning
import pickle, pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

with open("checkpoint_v1.pkl", "rb") as f: c1 = pickle.load(f)
with open("checkpoint_jaya_v1.pkl", "rb") as f: cj1 = pickle.load(f)

v1 = c1['df_test_summary_v1'].copy(); v1['Variation'] = 'V1 (3.5% real fraud, 30K test)'
v1 = v1.reset_index().rename(columns={'index': 'Model'})

jaya_row = {
    'Model': 'RXT-J Jaya', 'Variation': 'V1 (3.5% real fraud, 30K test)',
    'Accuracy':  accuracy_score(cj1['y_test_final'], cj1['rxt_j_pred_test_v1']),
    'Precision': precision_score(cj1['y_test_final'], cj1['rxt_j_pred_test_v1'], zero_division=0),
    'Recall':    recall_score(cj1['y_test_final'], cj1['rxt_j_pred_test_v1'], zero_division=0),
    'F1-Score':  f1_score(cj1['y_test_final'], cj1['rxt_j_pred_test_v1'], zero_division=0),
    'AUC-ROC':   roc_auc_score(cj1['y_test_final'], cj1['rxt_j_prob_test_v1']),
}
v1_full = pd.concat([v1, pd.DataFrame([jaya_row])], ignore_index=True)

pd.set_option('display.float_format', '{:.4f}'.format)
print("="*90)
print("V1 SUMMARY (baseline + DL + Jaya)")
print("="*90)
print(v1_full.to_string(index=False))

def threshold_tune(name, y_true, y_prob):
    print(f"\n{'='*70}\nTHRESHOLD TUNING — {name}\n{'='*70}")
    print(f"{'Threshold':<12} {'Accuracy':>10} {'Precision':>10} {'Recall':>8} {'F1':>8} {'FalseAlarms':>12}")
    best_t, best_f1 = 0.5, 0
    for t in [0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9]:
        preds = (y_prob > t).astype(int)
        acc, prec, rec, f1 = (accuracy_score(y_true,preds), precision_score(y_true,preds,zero_division=0),
                               recall_score(y_true,preds,zero_division=0), f1_score(y_true,preds,zero_division=0))
        fa = confusion_matrix(y_true, preds)[0][1]
        if f1 > best_f1: best_f1, best_t = f1, t
        print(f"  {t:<10} {acc*100:>9.2f}% {prec*100:>9.2f}% {rec*100:>7.2f}% {f1*100:>7.2f}% {fa:>11,}")
    print(f"Best threshold: {best_t} | Best F1: {best_f1*100:.2f}%")
    return best_t, best_f1

threshold_tune("RXT-J Jaya V1", cj1['y_test_final'], cj1['rxt_j_prob_test_v1'])
threshold_tune("ResNeXt-GRU + Attention V1", c1['y_test_final'], c1['att_prob_test'])

In [ ]:
# SHAP summary plot for RXT-J Jaya
import numpy as np
import shap

try:
    print("Trying GradientExplainer...")
    explainer_jaya = shap.GradientExplainer(rxt_j_model_v1, background)
    shap_values_jaya = explainer_jaya.shap_values(explain_sample)
    shap_values_jaya = shap_values_jaya[0] if isinstance(shap_values_jaya, list) else np.squeeze(shap_values_jaya, axis=-1)
    print("GradientExplainer worked.")
except Exception as e:
    print("GradientExplainer failed:", e)
    print("Falling back to model-agnostic Explainer (slower)...")
    predict_fn = lambda x: rxt_j_model_v1.predict(x, verbose=0).flatten()
    masker = shap.maskers.Independent(background, max_samples=50)
    explainer_jaya = shap.Explainer(predict_fn, masker)
    shap_values_jaya = explainer_jaya(explain_sample[:50]).values
    explain_sample = explain_sample[:50]

shap.summary_plot(shap_values_jaya, explain_sample, feature_names=feature_names_v1, max_display=20, show=True)

In [ ]:
# error analysis: false negatives / false positives by model
import pickle, numpy as np, pandas as pd
from sklearn.metrics import confusion_matrix

with open("checkpoint_v1.pkl", "rb") as f:
    c1 = pickle.load(f)
with open("checkpoint_jaya_v1.pkl", "rb") as f:
    cj1 = pickle.load(f)

X_test_final = np.load("X_test_final_v1.npy")
y_test_final = c1['y_test_final']

try:
    feature_names_v1
except NameError:
    feature_names_v1 = [f"feature_{i}" for i in range(X_test_final.shape[1])]

df_test = pd.DataFrame(X_test_final, columns=feature_names_v1)
df_test['y_true'] = y_test_final

models_to_analyze = {
    'RXT-J Jaya': cj1['rxt_j_pred_test_v1'],
    'ResNeXt-GRU + Attention': c1['att_pred_test'],
}

for model_name, y_pred in models_to_analyze.items():
    print("="*70)
    print(f"ERROR ANALYSIS — {model_name} (V1, 3.5% real fraud, 30K test)")
    print("="*70)

    tn, fp, fn, tp = confusion_matrix(y_test_final, y_pred).ravel()
    print(f"True Positives (caught fraud):    {tp:>6,}")
    print(f"False Negatives (missed fraud):   {fn:>6,}  <- costliest errors")
    print(f"False Positives (false alarms):   {fp:>6,}")
    print(f"True Negatives (correct legit):   {tn:>6,}")
    print(f"Miss rate (FN / actual fraud):    {fn/(fn+tp)*100:.2f}%")
    print(f"False alarm rate (FP / actual legit): {fp/(fp+tn)*100:.2f}%")

    df_test['pred'] = y_pred
    fn_rows = df_test[(df_test['y_true'] == 1) & (df_test['pred'] == 0)]
    fp_rows = df_test[(df_test['y_true'] == 0) & (df_test['pred'] == 1)]
    tp_rows = df_test[(df_test['y_true'] == 1) & (df_test['pred'] == 1)]

    if 'TransactionAmt' in feature_names_v1:
        print(f"\nTransactionAmt comparison:")
        print(f"  Missed fraud (FN) avg amount:  {fn_rows['TransactionAmt'].mean():.2f}")
        print(f"  Caught fraud (TP) avg amount:  {tp_rows['TransactionAmt'].mean():.2f}")
        print(f"  False alarms (FP) avg amount:  {fp_rows['TransactionAmt'].mean():.2f}")
        print(f"\nTop 5 highest-value MISSED frauds (costliest mistakes):")
        print(fn_rows.nlargest(5, 'TransactionAmt')[['TransactionAmt']].to_string())

    print()

In [ ]:
# extract per-epoch training time from saved cell outputs
import json, re

def get_epoch_times(text):
    times = re.findall(r'\x1b\[1m(\d+)s\x1b\[0m', text)
    return [int(t) for t in times]

def get_output_text(cell):
    return ''.join(''.join(o.get('text', [])) for o in cell.get('outputs', []))

def find_model_cells(nb_path, keywords=('rxt_model.fit', 'rxt_att_model.fit', 'rxt_j_model')):
    nb = json.load(open(nb_path, encoding='utf-8'))
    print("="*20, nb_path, "="*20)
    for i, cell in enumerate(nb['cells']):
        src = ''.join(cell.get('source', []))
        for kw in keywords:
            if kw in src:
                print(i, '|', kw, '|', src.split(chr(10))[0][:70])
    print()

def extract_training_times(nb_path, model_cells, outlier_threshold=200):
    nb = json.load(open(nb_path, encoding='utf-8'))
    results = {}
    print("="*20, nb_path, "="*20)
    for idx, name in model_cells:
        text = get_output_text(nb['cells'][idx])
        times = get_epoch_times(text)
        raw_total = sum(times)
        clean_total = sum(t for t in times if t < outlier_threshold)
        n_outliers = sum(1 for t in times if t >= outlier_threshold)
        results[name] = {'epochs': len(times), 'raw_total_s': raw_total,
                          'clean_total_s': clean_total, 'outlier_epochs': n_outliers}
        print(f"{name:<28} epochs={len(times):>3} | raw={raw_total/60:>6.1f} min | "
              f"clean={clean_total/60:>6.1f} min | outliers={n_outliers}")
    print()
    return results

v1_times = extract_training_times("IEEE-CIS-SET1.ipynb", [
    (8, "ResNeXt-GRU"), (11, "ResNeXt-GRU + Attention"), (34, "RXT-J Jaya")
])

In [ ]:
# prep dashboard demo artifacts (model + sample rows) from disk
import pickle
import numpy as np
import tensorflow as tf

model = tf.keras.models.load_model("dashboard_model.keras")

X_test_final = np.load("X_test_final_v1.npy")
y_test_final = np.load("y_test_final_v1.npy")

demo_idx = np.random.choice(len(X_test_final), 1000, replace=False)
np.save("dashboard_demo_X.npy", X_test_final[demo_idx])
np.save("dashboard_demo_y.npy", y_test_final[demo_idx])

print("Dashboard demo set updated to 1000 rows.")
print("Model, background, and feature files already exist from before — no need to resave those.")